In [42]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import StandardScaler
import pandas as pd 
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from optuna.visualization import plot_intermediate_values
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [5]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url,names=columns)

In [7]:
df.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [9]:
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

In [10]:
df.fillna(df.mean(),inplace=True)

In [11]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

In [14]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=42,stratify=y)

In [16]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators',50,200)
    max_depth = trial.suggest_int('max_depth',3,20)

    model = RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth,random_state=42)
    score = cross_val_score(model,X_train,y_train,scoring='accuracy',cv=3).mean()
    return score

In [19]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials=50)

[I 2026-03-11 01:02:18,868] A new study created in memory with name: no-name-4b8b1170-88dc-4ab6-83ef-40db2b5bd29c
[I 2026-03-11 01:02:19,414] Trial 0 finished with value: 0.7655348318189065 and parameters: {'n_estimators': 122, 'max_depth': 20}. Best is trial 0 with value: 0.7655348318189065.
[I 2026-03-11 01:02:20,086] Trial 1 finished with value: 0.7720229555236728 and parameters: {'n_estimators': 153, 'max_depth': 12}. Best is trial 1 with value: 0.7720229555236728.
[I 2026-03-11 01:02:20,469] Trial 2 finished with value: 0.7655188904830225 and parameters: {'n_estimators': 91, 'max_depth': 11}. Best is trial 1 with value: 0.7720229555236728.
[I 2026-03-11 01:02:21,158] Trial 3 finished with value: 0.7687868643392317 and parameters: {'n_estimators': 172, 'max_depth': 17}. Best is trial 1 with value: 0.7720229555236728.
[I 2026-03-11 01:02:21,980] Trial 4 finished with value: 0.7671528774111271 and parameters: {'n_estimators': 199, 'max_depth': 14}. Best is trial 1 with value: 0.77202

In [29]:
print('best accuracy',study.best_trial.value)
print('best parameters',study.best_trial.params)

best accuracy 0.7720468675274988
best parameters {'n_estimators': 164, 'max_depth': 15}


In [32]:
best_model = RandomForestClassifier(**study.best_trial.params,random_state=42)
best_model.fit(X_train,y_train)
y_pred = best_model.predict(X_test)
accuracy_score(y_test,y_pred)

0.7532467532467533

In [ ]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [37]:
plot_optimization_history(study).show()

In [38]:
plot_parallel_coordinate(study).show()

In [39]:
plot_slice(study).show()

In [40]:
plot_contour(study).show()

In [41]:
plot_param_importances(study).show()

In [46]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [47]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-03-11 01:53:31,848] A new study created in memory with name: no-name-a0f1fb78-d291-4641-9a60-f225aba14178
[I 2026-03-11 01:53:32,956] Trial 0 finished with value: 0.7393910409692332 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 82, 'learning_rate': 0.015455561666084004, 'max_depth': 19, 'min_samples_split': 7, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.7393910409692332.
[I 2026-03-11 01:53:34,580] Trial 1 finished with value: 0.7720149848557308 and parameters: {'classifier': 'RandomForest', 'n_estimators': 279, 'max_depth': 9, 'min_samples_split': 5, 'min_samples_leaf': 3, 'bootstrap': True}. Best is trial 1 with value: 0.7720149848557308.
[I 2026-03-11 01:53:38,914] Trial 2 finished with value: 0.7459509006854774 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 245, 'learning_rate': 0.04560937648399716, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 4}. Best is trial 1 with value: 0.7720149848557308.
[I 2026-03-11

In [48]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.2720979465425343, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7866889845369043


In [49]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.739391,2026-03-11 01:53:31.850300,2026-03-11 01:53:32.955990,0 days 00:00:01.105690,NaN,NaN,GradientBoosting,NaN,NaN,0.015456,19.0,4.0,7.0,82.0,COMPLETE
1,1,0.772015,2026-03-11 01:53:32.956930,2026-03-11 01:53:34.580842,0 days 00:00:01.623912,NaN,True,RandomForest,NaN,NaN,NaN,9.0,3.0,5.0,279.0,COMPLETE
2,2,0.745951,2026-03-11 01:53:34.582273,2026-03-11 01:53:38.914052,0 days 00:00:04.331779,NaN,NaN,GradientBoosting,NaN,NaN,0.045609,12.0,4.0,2.0,245.0,COMPLETE
3,3,0.765519,2026-03-11 01:53:38.915335,2026-03-11 01:53:39.943617,0 days 00:00:01.028282,NaN,True,RandomForest,NaN,NaN,NaN,13.0,2.0,9.0,186.0,COMPLETE
4,4,0.692197,2026-03-11 01:53:39.944932,2026-03-11 01:53:41.388470,0 days 00:00:01.443538,NaN,NaN,GradientBoosting,NaN,NaN,0.258508,15.0,1.0,4.0,82.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.767145,2026-03-11 01:54:02.282813,2026-03-11 01:54:02.315124,0 days 00:00:00.032311,0.358685,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.750813,2026-03-11 01:54:02.316642,2026-03-11 01:54:02.359259,0 days 00:00:00.042617,0.122679,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.786689,2026-03-11 01:54:02.360787,2026-03-11 01:54:02.388168,0 days 00:00:00.027381,0.199522,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.780169,2026-03-11 01:54:02.389336,2026-03-11 01:54:02.413356,0 days 00:00:00.024020,0.101509,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [50]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 78
GradientBoosting    11
RandomForest        11
Name: count, dtype: int64

In [51]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.746410
RandomForest        0.762980
SVM                 0.773088
Name: value, dtype: float64

In [52]:
plot_param_importances(study).show()